# 从混合文件夹中识别「服务报告」——Python Notebook 实战

这个 Notebook 按**步骤顺序**运行，适合在本地 Windows（也支持 macOS / Linux）处理工程维修档案。

**输入**：一个目录，混合包含 JPG/PNG/TIFF、PDF、XLSX、Markdown、TXT/CSV、DOCX/PPTX，以及 ZIP/RAR（支持嵌套压缩包）。  
**输出**：归档解压文件、所有可识别文件的 UTF-8 纯文本、服务报告的原文件和文本、需人工复核文件、CSV 清单和错误日志。

处理链路：

```text
input_files/
  ├─ 维修记录.zip ────┐
  ├─ 现场照片.jpg     │
  ├─ Service.pdf     ├──> 解压归档 ──> 按文件类型提取文字 ──> 关键词分类
  └─ 明细.xlsx       │                                         ├─ service_reports
                     ┘                                         ├─ review_candidates
                                                               └─ all_files_index.csv
```

> **边界说明**：这里提取的是**可搜索文字**（包括图片上的 OCR 文字），不会自动理解纯照片里设备的物理缺陷；扫描质量、手写体、加密文件或复杂 Excel 图表可能无法完整提取。分类是**可复核的规则法**，不是训练好的 AI 分类模型。

> **安全说明**：不要处理不可信的超大压缩包；本 Notebook 内置了 ZIP/RAR 路径穿越过滤、单档与总量限制和最大嵌套层级，但无法保证对所有恶意输入完全防护。原始数据只在本机处理。

## 第 0 步：准备环境（只需安装一次）

建议 Python 3.10+，用 VS Code 的 Jupyter 扩展或 JupyterLab 打开本文件。先执行下一格安装 Python 包。**安装完成后如导入报错，请重启 Notebook kernel**。

OCR 需要**额外安装 Tesseract 桌面程序**；仅 `pip install pytesseract` 并不足够。

- **Windows**：安装 Tesseract OCR，将 `tesseract.exe` 路径填写到后面的 `TESSERACT_CMD`（常见：`C:\Program Files\Tesseract-OCR\tesseract.exe`）。如需识别中文，请安装 `chi_sim` 语言包。
- **macOS**：可用 Homebrew 安装 `tesseract`，中文语言包需要另外安装。
- **Ubuntu/Debian**：常用 `sudo apt install tesseract-ocr tesseract-ocr-chi-sim`。
- **RAR 解压**：Python 包 `rarfile` 还依赖外部解压工具（例如 7-Zip、unrar/unar），Windows 可安装 7-Zip 并将它的程序目录加进 PATH。没有这个程序时 ZIP 仍可正常工作，但 RAR 会记录错误。详见 [rarfile 文档](https://rarfile.readthedocs.io/en/latest/)。

最初测试可以先只放 MD/TXT/PDF 等少量文件。

In [ ]:
%pip install -q pymupdf pytesseract Pillow openpyxl rarfile python-docx python-pptx

## 第 1 步：导入模块 + 设定路径

**必须修改 `SOURCE_DIR`**：指向你的原始文件夹；路径前面的 `r` 对 Windows 反斜杠很重要。例如 `Path(r"D:\ServiceArchive")`。运行结果输出到原目录的**旁边**，不会删除或移动原文件，每次启动 Notebook 会产生一个带时间戳的新输出目录。

In [ ]:
from pathlib import Path
from datetime import datetime
from collections import Counter, deque
from zipfile import ZipFile
from PIL import Image, ImageOps, ImageSequence
from docx import Document
from pptx import Presentation
from openpyxl import load_workbook
import csv
import hashlib
import json
import os
import re
import shutil
import stat
import traceback

import pymupdf
import pytesseract
import rarfile

# 修改为原始文件夹（注意不是某个具体文件）
SOURCE_DIR = Path(r"./input_files").expanduser().resolve()

# 新目录默认放在 SOURCE_DIR 旁边，保证不会扫描自身的输出
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
OUTPUT_DIR = SOURCE_DIR.parent / "service_report_output" / RUN_ID
UNPACK_DIR = OUTPUT_DIR / "01_unpacked_archives"
TEXT_DIR = OUTPUT_DIR / "02_all_text"
REPORT_DIR = OUTPUT_DIR / "03_service_reports"
REVIEW_DIR = OUTPUT_DIR / "04_review_candidates"
LOG_DIR = OUTPUT_DIR / "05_logs"

# 英文：eng；简体中文+英文：eng+chi_sim（需要提前安装对应语言包）
OCR_LANG = "eng+chi_sim"
# 示例: r"C:\Program Files\Tesseract-OCR\tesseract.exe"；在 PATH 中可保持 None
TESSERACT_CMD = None
# 扫描版 PDF 如果某一页几乎提取不到文字，则自动 OCR
OCR_PDF_ALWAYS = False  # 改为 True 将对所有 PDF 页面 OCR，速度更慢
OCR_PDF_MIN_CHARS = 30
OCR_DPI = 200

# 处理限制：避免误操作/压缩炸弹。视本地磁盘调整
MAX_ARCHIVE_DEPTH = 3
MAX_FILES_PER_ARCHIVE = 10000
MAX_ARCHIVE_TOTAL_MB = 500
MAX_SINGLE_FILE_MB = 250
MAX_TEXT_CHARS = 2_000_000

if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f"输入文件夹不存在: {SOURCE_DIR}。请先创建文件夹或修改 SOURCE_DIR")
if SOURCE_DIR == OUTPUT_DIR or SOURCE_DIR in OUTPUT_DIR.parents:
    raise ValueError("输出目录不能位于输入目录内，否则容易重复扫描")

for p in (UNPACK_DIR, TEXT_DIR, REPORT_DIR, REVIEW_DIR, LOG_DIR):
    p.mkdir(parents=True, exist_ok=True)

if TESSERACT_CMD:
    pytesseract.pytesseract.tesseract_cmd = TESSERACT_CMD

print("输入目录:", SOURCE_DIR)
print("输出目录:", OUTPUT_DIR)
print("找到的顶层文件数（包含子文件夹）:", sum(1 for p in SOURCE_DIR.rglob("*") if p.is_file()))

## 第 2 步：检查 OCR 和 RAR 环境

如果检测不到 `tesseract`，则图片以及扫描件 OCR 会报错，但其他文件仍可以处理；请先安装 Tesseract 再继续。中文档案建议检查是否同时有 `eng` 和 `chi_sim`。RAR 后端未找到时，RAR 会进入错误日志。

In [ ]:
print("Tesseract:", shutil.which("tesseract") or TESSERACT_CMD or "未检测到，请配置安装路径")
try:
    langs = pytesseract.get_languages(config="")
    print("OCR 已安装的语言:", langs)
    required = OCR_LANG.split("+")
    missing = [x for x in required if x not in langs]
    if missing:
        print("⚠️ 缺少 OCR 语言:", missing, "→ 可先将 OCR_LANG 改为 'eng'")
except Exception as exc:
    print("⚠️ OCR 程序不可用:", exc)

print("RAR 外部解压程序:", [(name, shutil.which(name)) for name in ("unrar", "unar", "7z", "7zz", "bsdtar") if shutil.which(name)])

## 第 3 步：解压 ZIP / RAR（含嵌套）

此步骤只**新建解压后的副本**，不会移动原文件。采用流式解压，并拒绝危险路径（例如 `../../windows/...`）、符号链接和过大成员。密码保护或格式损坏的归档会写入错误日志，不会令整个批处理停止。

例：`input_files/客户A/记录.zip` 会在 `01_unpacked_archives/客户A/记录.zip__contents/` 下展开。

In [ ]:
ARCHIVE_SUFFIXES = {".zip", ".rar"}
archive_results = []
extracted_count = 0


def source_label(path: Path) -> str:
    """用于索引的原始来源，压缩包用 ! 表示内部文件。"""
    path = path.resolve()
    if path.is_relative_to(SOURCE_DIR):
        return path.relative_to(SOURCE_DIR).as_posix()
    rel = path.relative_to(UNPACK_DIR).as_posix()
    return rel.replace(".zip__contents", ".zip!").replace(".rar__contents", ".rar!")


def safe_member_relative(raw_name: str):
    """不信任压缩文件里的路径；不允许绝对路径、..、盘符。"""
    norm = raw_name.replace("\\", "/")
    if norm.startswith("/") or norm.startswith("//"):
        return None
    parts = [p for p in norm.split("/") if p not in ("", ".")]
    if not parts or any(p == ".." for p in parts):
        return None
    if re.match(r"^[A-Za-z]:", parts[0]):
        return None
    return Path(*parts)


def copy_limited(src, dst_path: Path, max_bytes: int):
    """实际读取字节数也设上限，防止压缩包错误标称长度。"""
    written = 0
    dst_path.parent.mkdir(parents=True, exist_ok=True)
    try:
        with dst_path.open("wb") as dst:
            while True:
                chunk = src.read(1024 * 1024)
                if not chunk:
                    break
                written += len(chunk)
                if written > max_bytes:
                    raise ValueError("实际解压大小超出允许的限制")
                dst.write(chunk)
    except Exception:
        dst_path.unlink(missing_ok=True)
        raise
    return written


def target_for_archive(archive: Path) -> Path:
    if archive.is_relative_to(SOURCE_DIR):
        rel = archive.relative_to(SOURCE_DIR)
    else:
        rel = archive.relative_to(UNPACK_DIR)
    return UNPACK_DIR / rel.parent / (rel.name + "__contents")


def unpack_archive(archive: Path, depth: int):
    global extracted_count
    target = target_for_archive(archive)
    total_budget = MAX_ARCHIVE_TOTAL_MB * 1024 * 1024
    single_budget = MAX_SINGLE_FILE_MB * 1024 * 1024
    copied = 0
    count = 0
    target.mkdir(parents=True, exist_ok=True)

    if archive.suffix.lower() == ".zip":
        with ZipFile(archive) as zf:
            entries = zf.infolist()
            if len(entries) > MAX_FILES_PER_ARCHIVE:
                raise ValueError("ZIP 文件数量超限")
            if sum(max(0, i.file_size) for i in entries) > total_budget:
                raise ValueError("ZIP 解压总大小超限")
            for info in entries:
                if info.is_dir() or stat.S_ISLNK(info.external_attr >> 16):
                    continue
                rel = safe_member_relative(info.filename)
                if rel is None or info.file_size > single_budget:
                    continue
                dst = target / rel
                budget = min(single_budget, total_budget - copied)
                if budget <= 0:
                    raise ValueError("ZIP 实际解压总大小超限")
                with zf.open(info) as src:
                    copied += copy_limited(src, dst, budget)
                count += 1
    else:
        with rarfile.RarFile(archive) as rf:
            entries = rf.infolist()
            if len(entries) > MAX_FILES_PER_ARCHIVE:
                raise ValueError("RAR 文件数量超限")
            if sum(max(0, i.file_size) for i in entries) > total_budget:
                raise ValueError("RAR 解压总大小超限")
            for info in entries:
                if info.isdir():
                    continue
                is_link = getattr(info, "is_symlink", None)
                if callable(is_link) and is_link():
                    continue
                rel = safe_member_relative(info.filename)
                if rel is None or info.file_size > single_budget:
                    continue
                dst = target / rel
                budget = min(single_budget, total_budget - copied)
                if budget <= 0:
                    raise ValueError("RAR 实际解压总大小超限")
                with rf.open(info) as src:
                    copied += copy_limited(src, dst, budget)
                count += 1
    extracted_count += count
    return target, count


archive_queue = deque((p, 1) for p in sorted(SOURCE_DIR.rglob("*"))
                      if p.is_file() and not p.is_symlink() and p.suffix.lower() in ARCHIVE_SUFFIXES)
seen_archives = set()

while archive_queue:
    archive, depth = archive_queue.popleft()
    if archive in seen_archives:
        continue
    seen_archives.add(archive)
    label = source_label(archive)
    if depth > MAX_ARCHIVE_DEPTH:
        archive_results.append({"source": label, "status": "skipped_depth", "files": 0, "error": "嵌套层级超限"})
        continue
    try:
        target, count = unpack_archive(archive, depth)
        archive_results.append({"source": label, "status": "ok", "files": count, "error": ""})
        for sub in sorted(target.rglob("*")):
            if sub.is_file() and not sub.is_symlink() and sub.suffix.lower() in ARCHIVE_SUFFIXES:
                archive_queue.append((sub, depth + 1))
    except Exception as exc:
        archive_results.append({"source": label, "status": "error", "files": 0, "error": str(exc)})

print("找到压缩包:", len(archive_results), "解压后的文件总数:", extracted_count)
print("压缩包状态:", dict(Counter(r["status"] for r in archive_results)))
for item in archive_results[:10]:
    print(item)

## 第 4 步：按类型将文件转换成 UTF-8 文本

- JPG、PNG、TIFF → Tesseract OCR；TIFF 可处理多页。
- PDF → 优先提取原生文本；几乎没有文字的页用 OCR。
- XLSX/XLSM → 每个工作表的单元格，保留行列顺序；读取公式的**已缓存结果**，不执行 Excel 公式。
- MD/TXT/CSV/JSON → 尝试 UTF-8、UTF-16、GB18030 等编码。
- DOCX、PPTX → 提取正文/表格文字、幻灯片文字。

这一步**只提取文字，不破坏原文件**。Excel 内嵌图片、PPT 图片里的文字不会在首版中被 OCR；可以后续扩展。

In [ ]:
SUPPORTED_SUFFIXES = {
    ".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp",
    ".pdf", ".xlsx", ".xlsm", ".md", ".markdown", ".txt", ".csv", ".tsv", ".json",
    ".docx", ".pptx"
}


def read_text_guess(path: Path) -> str:
    raw = path.read_bytes()
    for encoding in ("utf-8-sig", "utf-16", "gb18030", "big5", "cp1252"):
        try:
            return raw.decode(encoding)
        except UnicodeDecodeError:
            continue
    return raw.decode("utf-8", errors="replace")


def extract_image(path: Path) -> str:
    result = []
    with Image.open(path) as im:
        for index, frame in enumerate(ImageSequence.Iterator(im), start=1):
            rgb = ImageOps.exif_transpose(frame.copy()).convert("RGB")
            recognized = pytesseract.image_to_string(rgb, lang=OCR_LANG)
            result.append(f"[图片第 {index} 帧]\n{recognized.strip()}")
    return "\n\n".join(result)


def extract_pdf(path: Path) -> str:
    sections = []
    with pymupdf.open(path) as document:
        for page_number, page in enumerate(document, start=1):
            raw_text = page.get_text("text", sort=True)
            if OCR_PDF_ALWAYS or len(raw_text.strip()) < OCR_PDF_MIN_CHARS:
                pix = page.get_pixmap(dpi=OCR_DPI, alpha=False)
                image = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
                ocr_text = pytesseract.image_to_string(image, lang=OCR_LANG)
                # 默认只补充信息：避免原生文本与 OCR 文本大段重复
                if OCR_PDF_ALWAYS and raw_text.strip():
                    raw_text += "\n[OCR 再识别]\n" + ocr_text
                else:
                    raw_text = ocr_text or raw_text
            sections.append(f"[PDF 第 {page_number} 页]\n{raw_text.strip()}")
    return "\n\n".join(sections)


def extract_xlsx(path: Path) -> str:
    lines = []
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        for sheet in workbook.worksheets:
            lines.append(f"[工作表: {sheet.title}]")
            for row in sheet.iter_rows(values_only=True):
                if all(value is None for value in row):
                    continue
                lines.append("\t".join("" if value is None else str(value) for value in row))
            lines.append("")
    finally:
        workbook.close()
    return "\n".join(lines)


def extract_docx(path: Path) -> str:
    doc = Document(path)
    parts = [p.text for p in doc.paragraphs if p.text.strip()]
    for index, table in enumerate(doc.tables, start=1):
        parts.append(f"[表格 {index}]")
        for row in table.rows:
            parts.append("\t".join(c.text.replace("\n", " ") for c in row.cells))
    return "\n".join(parts)


def extract_pptx(path: Path) -> str:
    pres = Presentation(path)
    parts = []
    for idx, slide in enumerate(pres.slides, start=1):
        parts.append(f"[幻灯片 {idx}]")
        for shape in slide.shapes:
            if shape.has_text_frame:
                parts.append(shape.text)
            if shape.has_table:
                for row in shape.table.rows:
                    parts.append("\t".join(cell.text for cell in row.cells))
    return "\n".join(parts)


def extract_any_text(path: Path) -> str:
    ext = path.suffix.lower()
    if ext == ".pdf":
        return extract_pdf(path)
    if ext in {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}:
        return extract_image(path)
    if ext in {".xlsx", ".xlsm"}:
        return extract_xlsx(path)
    if ext in {".md", ".markdown", ".txt", ".csv", ".tsv", ".json"}:
        return read_text_guess(path)
    if ext == ".docx":
        return extract_docx(path)
    if ext == ".pptx":
        return extract_pptx(path)
    raise ValueError(f"不支持的文件类型: {ext}")


def text_output_name(label: str, original_stem: str) -> str:
    unique = hashlib.sha256(label.encode("utf-8")).hexdigest()[:12]
    safe = re.sub(r'[<>:"/\\|?*\x00-\x1f]+', "_", original_stem)[:65] or "document"
    return f"{safe}__{unique}.txt"

## 第 5 步：批量提取、保存 `02_all_text` 和日志

一次只处理一种文件；单个文件损坏不会中断整个目录。索引保存 `source`（真实来源，如 `目录/zip文件.zip!/内部文件.pdf`）、`status`、字数、文本路径和错误原因。

首次建议只跑少量数据，确认 OCR 能工作后再放大规模。

In [ ]:
all_paths = [p for root in (SOURCE_DIR, UNPACK_DIR) for p in root.rglob("*")
             if p.is_file() and not p.is_symlink() and p.suffix.lower() not in ARCHIVE_SUFFIXES]
all_paths = sorted(set(all_paths))
records = []

for number, path in enumerate(all_paths, start=1):
    label = source_label(path)
    suffix = path.suffix.lower()
    record = {
        "source": label,
        "file_type": suffix,
        "bytes": path.stat().st_size,
        "status": "",
        "characters": 0,
        "text_path": "",
        "file_path": str(path),  # 内部使用，最终索引不必共享这列
        "error": "",
    }
    try:
        if suffix not in SUPPORTED_SUFFIXES:
            record["status"] = "unsupported"
            record["error"] = "暂不支持此格式；可扩展 extract_any_text()"
        elif record["bytes"] > MAX_SINGLE_FILE_MB * 1024 * 1024:
            record["status"] = "skipped_too_large"
            record["error"] = "文件太大"
        else:
            text = extract_any_text(path)
            if len(text) > MAX_TEXT_CHARS:
                text = text[:MAX_TEXT_CHARS] + "\n[提醒: 已达到文本长度限制，内容被截断]"
            out = TEXT_DIR / text_output_name(label, path.stem)
            out.write_text(text, encoding="utf-8")
            record["status"] = "ok" if text.strip() else "empty_text"
            record["characters"] = len(text)
            record["text_path"] = str(out)
    except Exception as exc:
        record["status"] = "error"
        record["error"] = f"{type(exc).__name__}: {exc}"
    records.append(record)
    if number % 20 == 0 or number == len(all_paths):
        print(f"已处理 {number}/{len(all_paths)}")

print("\n提取状态:", dict(Counter(row["status"] for row in records)))
print("文字内容保存在:", TEXT_DIR)

## 第 6 步：定义「服务报告」识别规则（可调整）

规则以**中文 + 英文**为目标，综合文件名、正文标题和字段：服务报告、维修报告、工作单号、设备、故障现象、处理措施、技术员等。与此同时，降低「发票、报价单、服务合同」的分数。

下面只实现**第一版高透明度规则**：高置信度自动保留，中等置信度放入 `04_review_candidates`。**建议人工核验一小批结果**，尤其确认没有漏掉报告。你可以改动关键词后重跑分类（无需重新 OCR）。

In [ ]:
TITLE_PATTERNS = [
    r"\b(?:field\s+)?service\s+report\b", r"\brepair\s+report\b",
    r"\btechnician(?:'s)?\s+report\b", r"\bmaintenance\s+report\b",
    r"(?:现场|售后|设备)?(?:维修|检修|保养|服务)报告", r"维修服务报告", r"服务记录报告",
]
NEGATIVE_PATTERNS = [
    r"\b(?:invoice|quotation|quote|purchase\s+order|service\s+agreement|service\s+contract)\b",
    r"发票|报价单|采购订单|服务合同|产品说明书|用户手册|销售合同",
]
SIGNAL_GROUPS = {
    "工单/报告编号": [r"\b(?:work\s*order|job\s*(?:no\.?|number|#)|report\s*(?:no\.?|number|#)|service\s*order)\b", r"工单(?:编号|号)?|报告编号|维修单号"],
    "客户/设备": [r"\b(?:customer|client|equipment|machine|model\s*(?:no\.?|number)?|serial\s*(?:no\.?|number))\b", r"客户名称|设备名称|设备型号|机型|序列号"],
    "服务日期": [r"\b(?:service\s+date|visit\s+date|inspection\s+date|date\s+of\s+service)\b", r"服务日期|维修日期|检修日期|到场时间"],
    "故障现象": [r"\b(?:fault\s+description|problem\s+description|issue\s+reported|failure\s+symptom|troubleshooting)\b", r"故障(?:现象|描述|原因)|问题描述|报修原因|无法启动|故障排查"],
    "检修/处理措施": [r"\b(?:work\s+performed|action\s+taken|corrective\s+action|repair\s+performed|parts\s+replaced)\b", r"维修过程|处理措施|维修内容|处理结果|更换(?:备件|零件|配件)|维修记录"],
    "技术员/签字": [r"\b(?:technician|field\s+engineer|service\s+engineer|customer\s+signature|engineer\s+signature)\b", r"维修工程师|服务工程师|技术员|维修人员|客户签字"],
    "测试/结果": [r"\b(?:test\s+result|inspection\s+result|repair\s+completed|resolution|root\s+cause)\b", r"测试结果|维修完成|修复完成|恢复正常|故障已排除|检修结论"],
}


def regex_found(patterns, text):
    return any(re.search(p, text, flags=re.IGNORECASE) for p in patterns)


def classify_report(source: str, body: str):
    first_section = body[:2500]
    file_name = source.split("/")[-1]
    title_in_name = regex_found(TITLE_PATTERNS, file_name.replace("_", " ").replace("-", " "))
    title_in_body = regex_found(TITLE_PATTERNS, first_section)
    negative_name = regex_found(NEGATIVE_PATTERNS, file_name)
    negative_first = regex_found(NEGATIVE_PATTERNS, first_section[:800])

    matched = [group for group, patterns in SIGNAL_GROUPS.items()
               if regex_found(patterns, body)]
    score = (7 if title_in_name else 0) + (7 if title_in_body else 0) + 2 * len(matched)
    score -= (7 if negative_name else 0) + (3 if negative_first else 0)

    if score >= 7 and (title_in_name or title_in_body) and not negative_name:
        label = "service_report"
    elif score >= 10 and len(matched) >= 4 and not negative_name:
        label = "service_report"
    elif score >= 4 and len(matched) >= 2:
        label = "review"
    else:
        label = "other"

    evidence = []
    if title_in_name:
        evidence.append("文件名含报告标题")
    if title_in_body:
        evidence.append("正文含报告标题")
    evidence.extend(matched)
    if negative_name or negative_first:
        evidence.append("出现非报告类关键词(扣分)")
    return {"label": label, "score": score, "evidence": "; ".join(evidence)}


def extract_snippets(body: str, max_matches=5, context=140) -> str:
    """保存支持判断的前几处上下文，而不只是一个分数。"""
    match_positions = []
    patterns = TITLE_PATTERNS + [p for group in SIGNAL_GROUPS.values() for p in group]
    for p in patterns:
        match = re.search(p, body, flags=re.IGNORECASE)
        if match and all(abs(match.start() - x) > 100 for x in match_positions):
            match_positions.append(match.start())
        if len(match_positions) >= max_matches:
            break
    snippets = []
    for pos in sorted(match_positions)[:max_matches]:
        snippets.append(body[max(0, pos-context):pos+context].replace("\n", " "))
    return "\n---\n".join(snippets) or body[:350].replace("\n", " ")

## 第 7 步：执行分类，复制报告原文件及文本

输出分两类：

- `03_service_reports`：**较高置信度**服务报告（含 `originals/` 和 `text/`）；
- `04_review_candidates`：可能相关，但需要人工判断的文件。

复制原文件时会附上稳定的来源 ID，避免不同压缩包里出现同名文件导致覆盖。

In [ ]:
def copy_selected_files(items, target_dir: Path):
    (target_dir / "originals").mkdir(parents=True, exist_ok=True)
    (target_dir / "text").mkdir(parents=True, exist_ok=True)
    for row in items:
        orig = Path(row["file_path"])
        text_path = Path(row["text_path"])
        stable_id = hashlib.sha256(row["source"].encode("utf-8")).hexdigest()[:12]
        safe_name = re.sub(r'[<>:"/\\|?*\x00-\x1f]+', "_", orig.stem)[:65] or "document"
        orig_copy = target_dir / "originals" / f"{safe_name}__{stable_id}{orig.suffix}"
        txt_copy = target_dir / "text" / f"{safe_name}__{stable_id}.txt"
        shutil.copy2(orig, orig_copy)
        shutil.copy2(text_path, txt_copy)
        row["selected_original"] = str(orig_copy)
        row["selected_text"] = str(txt_copy)

for row in records:
    row.update({"label": "not_classified", "score": "", "evidence": "", "snippet": "", "selected_original": "", "selected_text": ""})
    if row["status"] != "ok":
        continue
    text = Path(row["text_path"]).read_text(encoding="utf-8")
    decision = classify_report(row["source"], text)
    row.update(decision)
    if decision["label"] != "other":
        row["snippet"] = extract_snippets(text)

service_reports = [r for r in records if r["label"] == "service_report"]
review_candidates = [r for r in records if r["label"] == "review"]
copy_selected_files(service_reports, REPORT_DIR)
copy_selected_files(review_candidates, REVIEW_DIR)

print("服务报告:", len(service_reports))
print("需要人工复核:", len(review_candidates))
print("其他/暂不支持/处理失败:", len(records) - len(service_reports) - len(review_candidates))
for row in sorted(service_reports, key=lambda x: x["score"], reverse=True)[:8]:
    print(f" - [score={row['score']}] {row['source']} | {row['evidence']}")

## 第 8 步：导出 CSV 清单和错误日志

CSV 使用 `utf-8-sig`（便于中文 Windows Excel 直接打开）。索引可定位原始来源，服务报告索引还带有**匹配证据和上下文片段**。不要只看 `03_service_reports`：先抽样检查 `04_review_candidates` 和 `other` 以评估漏检。

In [ ]:
def write_csv(path, rows, columns):
    with path.open("w", encoding="utf-8-sig", newline="") as fp:
        writer = csv.DictWriter(fp, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(rows)

index_columns = ["source", "file_type", "bytes", "status", "characters", "label",
                 "score", "evidence", "snippet", "text_path", "selected_original", "selected_text", "error"]
write_csv(LOG_DIR / "all_files_index.csv", records, index_columns)
write_csv(LOG_DIR / "service_reports.csv", service_reports, index_columns)
write_csv(LOG_DIR / "review_candidates.csv", review_candidates, index_columns)
write_csv(LOG_DIR / "archive_log.csv", archive_results, ["source", "status", "files", "error"])
write_csv(LOG_DIR / "processing_issues.csv",
          [r for r in records if r["status"] not in {"ok", "empty_text"}], index_columns)

summary = {
    "input_directory": str(SOURCE_DIR),
    "output_directory": str(OUTPUT_DIR),
    "total_non_archive_files": len(records),
    "extraction_status": dict(Counter(r["status"] for r in records)),
    "classification_result": dict(Counter(r["label"] for r in records)),
    "archive_status": dict(Counter(r["status"] for r in archive_results)),
    "ocr_language": OCR_LANG,
}
(LOG_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("\n完整结果:", OUTPUT_DIR)
print("已识别的服务报告:", REPORT_DIR)
print("需要人工复核的文件:", REVIEW_DIR)
print("索引:", LOG_DIR / "all_files_index.csv")

## 第 9 步：手动查看几个匹配结果

下面展示前 5 份服务报告的标题、匹配依据和正文开头。修改分类规则后，重新执行第 7～9 步即可；不需要再执行解压和 OCR。

In [ ]:
for row in service_reports[:5]:
    print("=" * 70)
    print("文件:", row["source"])
    print("分数:", row["score"], "证据:", row["evidence"])
    print("正文片段:", row["snippet"][:800])
    print()
if not service_reports:
    print("当前没有高置信度结果。请检查 OCR 日志，并查看 review_candidates.csv，或微调 TITLE_PATTERNS / SIGNAL_GROUPS。")

## 第 10 步：如何提高准确率（下一阶段）

1. **先收集标注样本**：从 `all_files_index.csv` 随机选 30–50 份，人工标为「服务报告／不是／不确定」，观察假阳性/假阴性。
2. **扩展特征**：报告往往含设备型号、故障现象、故障原因、处理措施、技术员、日期；可以把文档中这些结构化字段抽到 JSON。
3. **混合页 PDF**：部分页面有可选中文本、部分是图片。首版采用逐页空白判断；如果怀疑丢了图片中的文字，把 `OCR_PDF_ALWAYS=True` 再跑第 5 步。
4. **更强分类器（可选）**：在本地搭建 embedding + 相似度检索，或使用本地 LLM（例如 Ollama）对 `review` 文档做二次判断，输出理由。模型判断也需要人工抽检。
5. **下一步做知识库**：建议把 `source`、报告日期、设备型号、故障、原因、维修措施存入 SQLite / PostgreSQL，再做全文检索或 RAG 问答。

**已知局限**：Excel / PPTX 内嵌图片未做 OCR；PDF 表格可能失去行列布局；旧版 `.xls`、`.doc`、加密 ZIP/RAR 未支持；RAR 可能因缺少外部解压程序失败；非常大的文件可能因安全上限被跳过；规则分类不保证零漏检。

**提示**：文件可能包含客户名称、电话、设备序列号等私密信息；请将输出文件夹留在受控设备上，上传第三方 AI 服务前先做脱敏。